In [57]:
import warnings

from datamodules.data_classes import PotsdamVaihingen
from rasterio.errors import NotGeoreferencedWarning

warnings.filterwarnings("ignore", category=NotGeoreferencedWarning)

train_vaihingen_image_glob = "../Vaihingen_dataset/train/images/*.tif"
train_vaihingen_mask_glob = "../Vaihingen_dataset/train/masks/*.tif"
val_vaihingen_image_glob = "../Vaihingen_dataset/val/images/*.tif"
val_vaihingen_mask_glob = "../Vaihingen_dataset/val/masks/*.tif"

train_data = PotsdamVaihingen(
    image_glob=train_vaihingen_image_glob,
    mask_glob=train_vaihingen_mask_glob,
    reduce_mask=True,
)

val_data = PotsdamVaihingen(
    image_glob=val_vaihingen_image_glob,
    mask_glob=val_vaihingen_mask_glob,
    reduce_mask=True,
)

train_data[0]["image"].shape, train_data[0]["mask"].shape

(torch.Size([3, 2569, 1919]), torch.Size([1, 2569, 1919]))

In [2]:
len(train_data), len(val_data)

(26, 7)

In [1]:
import gc
from datasets import Dataset, Features, Array3D, Array2D
import torchio as tio


class HFDataset:
    def __init__(self, dataset):
        self.dataset = dataset

    def build_patches(
        self,
        output_path,
        patch_kwargs=None,
        shard_size=5000,
    ):
        all_images = []
        all_masks = []

        patch_kwargs["patch_size"][0]
        windows_params = dict(
            ph=patch_kwargs["patch_size"][0],
            pw=patch_kwargs["patch_size"][1],
            oh=patch_kwargs["overlap"][0],
            ow=patch_kwargs["overlap"][1],
        )

        shard_idx = 0
        for element in self.dataset:
            image = element["image"].unsqueeze(-1).squeeze(0)
            mask = element["mask"].unsqueeze(-1)
            if mask.ndim == 3:
                mask = mask.unsqueeze(0)

            subject = tio.Subject(
                image=tio.ScalarImage(tensor=image), mask=tio.LabelMap(tensor=mask)
            )
            sampler = tio.GridSampler(
                subject,
                patch_size=(windows_params["ph"], windows_params["pw"], 1),
                patch_overlap=(windows_params["oh"], windows_params["ow"], 0),
            )
            for patch in sampler:
                img_np = (
                    patch["image"][tio.DATA]
                    .squeeze(-1)
                    .contiguous()
                    .numpy()
                    .astype("float32")
                )
                msk_np = (
                    patch["mask"][tio.DATA]
                    .squeeze(-1)
                    .squeeze(0)
                    .contiguous()
                    .numpy()
                    .astype("uint8")
                )
                del patch

                all_images.append(img_np)
                all_masks.append(msk_np)

                if len(all_images) >= shard_size:
                    self._save_shard(
                        output_path, shard_idx, all_images, all_masks, windows_params
                    )
                    shard_idx += 1

                    all_images.clear()
                    all_masks.clear()

                    gc.collect()

            del subject, sampler
            gc.collect()

        if len(all_images) > 0:
            self._save_shard(
                output_path, shard_idx, all_images, all_masks, windows_params
            )
            all_images.clear()
            all_masks.clear()
            gc.collect()

    def build_full(self, output_path, shard_size=2):
        all_images = []
        all_masks = []

        shard_idx = 0

        for element in self.dataset:
            image = element["image"].squeeze(0)
            mask = element["mask"]

            all_images.append(image.numpy())
            all_masks.append(mask.squeeze(0).numpy())

            if len(all_images) >= shard_size:
                self._save_shard(
                    output_path,
                    shard_idx,
                    all_images,
                    all_masks,
                    windows_params=None,
                    include_features=False,
                )
                shard_idx += 1

                all_images.clear()
                all_masks.clear()

                gc.collect()

        gc.collect()

        if len(all_images) > 0:
            self._save_shard(
                output_path,
                shard_idx,
                all_images,
                all_masks,
                windows_params=None,
                include_features=False,
            )
            all_images.clear()
            all_masks.clear()
            gc.collect()

    def _save_shard(
        self,
        output_path,
        shard_idx,
        all_images,
        all_masks,
        windows_params,
        include_features=True,
    ):

        if include_features:
            features = Features(
                {
                    "image": Array3D(
                        shape=(3, windows_params["ph"], windows_params["pw"]),
                        dtype="float32",
                    ),
                    "mask": Array2D(
                        shape=(windows_params["ph"], windows_params["pw"]),
                        dtype="uint8",
                    ),
                }
            )

        dataset = Dataset.from_dict(
            {
                "image": all_images,
                "mask": all_masks,
            },
            features=features if include_features else None,
        )

        dataset.save_to_disk(f"{output_path}/shard_{shard_idx:03d}")
        del dataset
        gc.collect()

In [4]:
OUTPUT_DIR = "Vaihingen_HF"

training_output_path = f"../{OUTPUT_DIR}/vaihingen_train_patches-256x256"
validation_output_path = f"../{OUTPUT_DIR}/vaihingen_validation"

In [6]:
train_hf_dataset = HFDataset(train_data)
train_hf_dataset.build_patches(
    training_output_path,
    patch_kwargs=dict(patch_size=(256, 256), overlap=(0, 0)),
    shard_size=5000,
)

Saving the dataset (0/4 shards):   0%|          | 0/2256 [00:00<?, ? examples/s]

In [10]:
val_hf_dataset = HFDataset(val_data)
val_hf_dataset.build_full(validation_output_path, shard_size=10)

Saving the dataset (0/2 shards):   0%|          | 0/7 [00:00<?, ? examples/s]

In [8]:
import glob
from datasets import load_from_disk, concatenate_datasets

train_path = f"../{OUTPUT_DIR}/vaihingen_train_patches-256x256/*"


def load_shards_into_dataset(path):
    paths = [load_from_disk(p) for p in glob.glob(path)]

    vaihingen_train_dataset = concatenate_datasets(paths)
    return vaihingen_train_dataset


vaihingen_train_dataset = load_shards_into_dataset(train_path)
len(vaihingen_train_dataset)

2256

In [11]:
val_path = f"../{OUTPUT_DIR}/vaihingen_validation/*"
vaihingen_val_dataset = load_shards_into_dataset(val_path)
len(vaihingen_val_dataset)

7

In [12]:
len(vaihingen_train_dataset), len(vaihingen_val_dataset)

(2256, 7)

In [13]:
import warnings

from datamodules.data_classes import PotsdamVaihingen
from rasterio.errors import NotGeoreferencedWarning

warnings.filterwarnings("ignore", category=NotGeoreferencedWarning)

train_potsdam_image_glob = "../Potsdam_dataset/train/images/*.tif"
train_potsdam_mask_glob = "../Potsdam_dataset/train/masks/*.tif"

val_potsdam_image_glob = "../Potsdam_dataset/val/images/*.tif"
val_potsdam_mask_glob = "../Potsdam_dataset/val/masks/*.tif"

train_data = PotsdamVaihingen(
    image_glob=train_potsdam_image_glob,
    mask_glob=train_potsdam_mask_glob,
    reduce_mask=True,
)

val_data = PotsdamVaihingen(
    image_glob=val_potsdam_image_glob,
    mask_glob=val_potsdam_mask_glob,
    reduce_mask=True,
)

In [14]:
len(train_data), len(val_data)

(30, 8)

In [15]:
OUTPUT_DIR = "Potsdam_HF"

training_output_path = f"../{OUTPUT_DIR}/potsdam_train_patches-256x256"
validation_output_path = f"../{OUTPUT_DIR}/potsdam_validation"

train_hf_dataset = HFDataset(train_data)
train_hf_dataset.build_patches(
    training_output_path,
    patch_kwargs=dict(patch_size=(256, 256), overlap=(0, 0)),
    shard_size=3000,
)

Saving the dataset (0/6 shards):   0%|          | 0/3000 [00:00<?, ? examples/s]

Saving the dataset (0/6 shards):   0%|          | 0/3000 [00:00<?, ? examples/s]

Saving the dataset (0/6 shards):   0%|          | 0/3000 [00:00<?, ? examples/s]

Saving the dataset (0/6 shards):   0%|          | 0/3000 [00:00<?, ? examples/s]

Saving the dataset (0/6 shards):   0%|          | 0/3000 [00:00<?, ? examples/s]

Saving the dataset (0/4 shards):   0%|          | 0/2280 [00:00<?, ? examples/s]

In [16]:
val_hf_dataset = HFDataset(val_data)
val_hf_dataset.build_full(
    validation_output_path,
    shard_size=2,
)

Saving the dataset (0/2 shards):   0%|          | 0/2 [00:00<?, ? examples/s]

Saving the dataset (0/2 shards):   0%|          | 0/2 [00:00<?, ? examples/s]

Saving the dataset (0/2 shards):   0%|          | 0/2 [00:00<?, ? examples/s]

Saving the dataset (0/2 shards):   0%|          | 0/2 [00:00<?, ? examples/s]

In [18]:
train_path = f"../{OUTPUT_DIR}/potsdam_train_patches-256x256/*"
potsdam_train_dataset = load_shards_into_dataset(train_path)
print(len(potsdam_train_dataset))

val_path = f"../{OUTPUT_DIR}/potsdam_validation/*"
potsdam_val_dataset = load_shards_into_dataset(val_path)
len(potsdam_val_dataset)

17280


8

In [2]:
from torchgeo.datasets import LoveDA

train_data = LoveDA(
    root="../loveda_dataset",
    split="train",
    scene=["urban", "rural"],
)

val_data = LoveDA(
    root="../loveda_dataset",
    split="val",
    scene=["urban", "rural"],
)

In [3]:
len(train_data), len(val_data)

(2522, 1669)

In [4]:
train_data[0]["image"].shape, train_data[0]["mask"].shape

(torch.Size([3, 1024, 1024]), torch.Size([1024, 1024]))

In [5]:
OUTPUT_DIR = "LOVEDA_HF"

training_output_path = f"../{OUTPUT_DIR}/train_patches-256x256"
validation_output_path = f"../{OUTPUT_DIR}/validation"

train_hf_dataset = HFDataset(train_data)
train_hf_dataset.build_patches(
    training_output_path,
    patch_kwargs=dict(patch_size=(256, 256), overlap=(0, 0)),
    shard_size=3000,
)

Saving the dataset (0/6 shards):   0%|          | 0/3000 [00:00<?, ? examples/s]

Saving the dataset (0/6 shards):   0%|          | 0/3000 [00:00<?, ? examples/s]

Saving the dataset (0/6 shards):   0%|          | 0/3000 [00:00<?, ? examples/s]

Saving the dataset (0/6 shards):   0%|          | 0/3000 [00:00<?, ? examples/s]

Saving the dataset (0/6 shards):   0%|          | 0/3000 [00:00<?, ? examples/s]

...

Saving the dataset (0/6 shards):   0%|          | 0/3000 [00:00<?, ? examples/s]

Saving the dataset (0/6 shards):   0%|          | 0/3000 [00:00<?, ? examples/s]

Saving the dataset (0/6 shards):   0%|          | 0/3000 [00:00<?, ? examples/s]

Saving the dataset (0/6 shards):   0%|          | 0/3000 [00:00<?, ? examples/s]

Saving the dataset (0/3 shards):   0%|          | 0/1352 [00:00<?, ? examples/s]

In [ ]:
val_hf_dataset = HFDataset(val_data)
val_hf_dataset.build_full(
    validation_output_path,
    shard_size=2,
)